## Databricks Multi-Tool Agent Setup

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    "mlflow>=3.1"

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setup your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Define your MCP Server URLs

In [ ]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

custom_mcp_server_url = "YOUR-CUSTOM-MCP-SERVER-URL-GOES-HERE"

ai_search_mcp_url = (
    f"{workspace_host}/api/2.0/mcp/vector-search/"
    f"<catalog>/<shema>/<name>"
)

### Create and Execute the Agent

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)
from agents.mcp import MCPServerStreamableHttp

# Create an OpenAI-compatible client for Databricks
client = AsyncOpenAI(
    api_key=token,
    base_url=f"{workspace_host}/serving-endpoints"
)

# Configure the Databricks model
model = OpenAIChatCompletionsModel(
    model="databricks-claude-sonnet-4-5",
    openai_client=client
)

async with (
    MCPServerStreamableHttp(
        name="CarbonOps-MCP-Server",
        params={
            "url": custom_mcp_server_url,
            "headers": {
                "Authorization": f"Bearer {token}"
            }
        }
    ) as carbonops_mcp_server,

    MCPServerStreamableHttp(
        name="CarbonOps-AI-Search",
        params={
            "url": ai_search_mcp_url,
            "headers": {
                "Authorization": f"Bearer {token}"
            }
        }
    ) as carbonops_ai_search
):

    # Don't send Agents SDK traces to OpenAI
    set_tracing_disabled(True)
    
    # Create agent
    agent = Agent(
        name="Multi-Tool-Agent",
        instructions=(
            "You are the CarbonOps Sustainability Intelligence Assistant. "
            "Use Databricks AI Search to retrieve sustainability documents, "
            "policies, and operational knowledge. "
            "Use the CarbonOps MCP Server to predict emissions, assess risks, "
            "calculate carbon intensity, and simulate emissions reductions. "
            "Automatically select and combine tools based on user queries. "
            "Provide clear, accurate, and actionable insights grounded in "
            "retrieved information and tool results. "
            "Never fabricate data or predictions."
        ),
        model=model,
        mcp_servers=[carbonops_ai_search, carbonops_mcp_server]
    )

    # Run agent
    result = await Runner.run(
        agent,
        "How much carbon emissions can we expect in June 2026 if our factory consumes 50,000 kWh"
        "of electricity, 2,500 litres of diesel, 1,200 m³ of natural gas, and 500 kg of LPG," 
        "while producing 10,000 units over 500 operating hours at 32°C?"
    )

    print(result.final_output)